# ZemaVoice Stage 12 — Adult Amharic Voice Studio

Test **genuinely distinct available Amharic voice identities**, including women and men, and export WAV files for your movies. This experiment uses `edge-tts`, an **unofficial** interface to Microsoft's online speech service: internet access is required, service availability can change, and commercial/production usage requires checking applicable terms. For a durable ZemaVoice website/API, plan to migrate to an officially supported licensed TTS provider or owned models. **No Microsoft account key is required for this Colab experiment.**

The notebook discovers actual voices at runtime and only displays those advertised for Amharic (`am-ET`). It **does not promise a particular number** of distinct Amharic speakers. Speed and pitch variations of one voice are not new speaker identities.


In [ ]:
!pip -q install 'edge-tts>=7.0' 'ipywidgets>=8' 'soundfile>=0.12'


In [ ]:
import edge_tts, asyncio, os, re, subprocess, sys
import ipywidgets as widgets
from IPython.display import display,Audio
from pathlib import Path
print('Discovering available Amharic voices...')
voices=await edge_tts.list_voices()
amharic=[v for v in voices if v.get('Locale','').lower().startswith('am-et')]
for v in amharic: print(v.get('ShortName'), '|',v.get('Gender'), '|',v.get('FriendlyName'))
if not amharic: print('No Amharic voices returned. This online service may be unavailable; send this output to ChatGPT.')


In [ ]:
assert amharic,'No am-ET voices found; stop and send the preceding output.'
voice_names=[v['ShortName'] for v in amharic]
voice=widgets.Dropdown(options=voice_names,description='Voice:',layout=widgets.Layout(width='75%'))
text_input=widgets.Textarea(value='ሰላም! ወደ ዜማ ቮይስ እንኳን ደህና መጡ። ዛሬ አዲስ ታሪክ እናቀርባለን።',description='Amharic:',layout=widgets.Layout(width='95%',height='140px'))
rate=widgets.IntSlider(value=0,min=-25,max=25,step=5,description='Speed %:')
pitch=widgets.IntSlider(value=0,min=-10,max=10,step=2,description='Pitch Hz:')
display(widgets.VBox([voice,text_input,rate,pitch]))
print('Select an actual voice from the list. Adjust speed and pitch only if needed. Then run the next cell.')


In [ ]:
from IPython.display import Audio, display
from google.colab import files
raw=text_input.value.strip()
assert raw and len(raw)<=2000,'Enter 1 to 2000 characters.'
selected=voice.value
slug=re.sub(r'[^a-zA-Z0-9_-]','_',selected)
out_mp3=f'/content/zemavoice_stage12_{slug}.mp3'
out_wav=f'/content/zemavoice_stage12_{slug}.wav'
rate_str=f'{rate.value:+d}%'
pitch_str=f'{pitch.value:+d}Hz'
print('Generating:',selected,'speed:',rate_str,'pitch:',pitch_str)
communicate=edge_tts.Communicate(raw,selected,rate=rate_str,pitch=pitch_str)
await communicate.save(out_mp3)
result=subprocess.run(['ffmpeg','-y','-loglevel','error','-i',out_mp3,'-ar','24000','-ac','1',out_wav],capture_output=True,text=True)
if result.returncode: raise RuntimeError('Audio conversion failed: '+result.stderr[-1500:])
print('Saved WAV:',out_wav)
display(Audio(filename=out_wav))


In [ ]:
from google.colab import files
files.download(out_wav)


## Multi-character movie dialogue
Generate each character's line separately by changing the selected voice and text, then rerunning the generation cell. Import the WAV files into your existing Stage 7 synchronization workflow.

**If only one female and one male Amharic voice appear:** that is the actual number available from this provider in your session, not a bug. To obtain more genuinely different voices we will need additional licensed Amharic speaker models or consented recordings; changing pitch does not create a new person.

**Privacy:** Text entered here is sent to an external online speech service. Do not enter private scripts or personal data unless you accept that.
